# Chapter 3

## Summarizing a document bigger than the LLM’s context window

In [1]:
# Install required packages
!pip install langchain-google-genai google-generativeai

In [1]:
!pip install tiktoken notebook langchain langchain-google-genai google-generativeai langchain-community langchain-core langchain-text-splitters wikipedia docx2txt pypdf

INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 5.3 MB/s eta 0:00:00
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.7/362.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.4 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.1.38
    Uninstalling langchain-core-0.1.38:
      Successfully uninstalled langchain-core-0.1.38
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-te

In [2]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import os

In [4]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('APIKeyForLLMs')

In [5]:
llm = ChatGoogleGenerativeAI(google_api_key=GOOGLE_API_KEY, model="gemini-3-flash-preview")

In [6]:
print(llm.model)

gemini-3-flash-preview


In [7]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x:
        [
            {
                'chunk': text_chunk,
            }
            for text_chunk in
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [8]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

In [9]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x:
        {
            'summaries': '\n'.join([i['summary'] for i in x]),
        })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

In [10]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)

In [11]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [12]:
print(summary)

Driven by a spiritual attraction to the sea and a desire to escape melancholy, Ishmael decides to join a whaling voyage as a common sailor, attributing the choice to "Fate." Traveling from Manhattan to New Bedford on his way to Nantucket, he finds lodging at the "Spouter-Inn," owned by Peter Coffin. 

Because the inn is full, Ishmael is forced to share a bed with a mysterious harpooneer named Queequeg. Initially terrified by Queequeg’s appearance—a heavily tattooed "cannibal" who sells shrunken heads and performs pagan rituals—Ishmael’s anxiety leads to a brief confrontation. However, he quickly overcomes his prejudices, concluding that it is better to sleep with a "sober cannibal than a drunken Christian." The two form an unlikely bond, cemented when Ishmael wakes up in Queequeg’s affectionate embrace and observes the harpooneer’s eccentric but delicate morning routine, which includes shaving his face with a razor-sharp harpoon.


## Summarizing across documents

In [13]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [16]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [17]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [18]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

In [19]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful,
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [25]:
llm = ChatGoogleGenerativeAI(google_api_key=GOOGLE_API_KEY, model="gemini-3.1-pro-preview")

In [23]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary,
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)

        current_refined_summary = refine_chain.invoke(intermediate_step)

    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [26]:
full_summary = refine_summary(all_docs)
print(full_summary['final_summary'])

ChatGoogleGenerativeAIError: Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 141.494386ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-Fr